# M03B: Error Handling & Retry Logic

APIs fail. Networks timeout. Rate limits hit. Your production code needs to handle all of it.

**Topics:**
- Robust API wrapper with automatic retry
- Exponential backoff
- Fallback strategies for graceful degradation

---

## 🔧 Step 1: Setup

In [ ]:
import os
import time
from pathlib import Path
from dotenv import load_dotenv
import openai

load_dotenv(dotenv_path=Path("..") / ".env")

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

MODEL = "gpt-5-mini"


print("✅ Ready!")

---

## ⚠️ The "Broken" Helper Function

This version has NO error handling. It will crash on any API error.

We'll fix it step-by-step.

In [ ]:
def ask_openai_v0(prompt):
    """
    Version 0: NO error handling.
    This will crash on errors!
    """
    response = client.responses.create(
        model=MODEL,
        input=prompt
    )
    return response.output_text.strip()

print("⚠️ v0 loaded - no error handling!")

---

## 🎯 Common API Failure Modes

<div style="text-align: left; display: inline-block;">

| Type | Examples |
|------|----------|
| 🔒 Authentication | Invalid/expired API key |
| ⏱️ Rate Limits | Too many requests, quota exceeded |
| 🔌 Network | Timeout, connection failed |
| ☁️ Server | 500/503 errors, API down |
| 🚫 Invalid Request | Bad model name, prompt too long |

</div>

Without error handling, any of these crashes your app.

---

## 🔨 Phase 1: Basic Error Handling

Catch errors and return useful messages instead of crashing.

Let's build version 1 with basic error handling:

In [ ]:
def ask_openai_v1(prompt):
    """
    Version 1: Basic error handling.
    Catches errors and returns a dict with success/error info.
    """
    
    try:
        response = client.responses.create(
            model=MODEL,
            input=prompt
        )
        # ✅ NEW: Return dict with success flag
        return {"success": True, "data": response.output_text.strip()}
    
    # ✅ NEW: Catch specific OpenAI errors
    except openai.AuthenticationError as e:
        return {"success": False, "error": "Authentication failed", "details": str(e)}
    
    except openai.RateLimitError as e:
        return {"success": False, "error": "Rate limit exceeded", "details": str(e)}
    
    except openai.APIConnectionError as e:
        return {"success": False, "error": "Network connection failed", "details": str(e)}
    
    except openai.BadRequestError as e:
        return {"success": False, "error": "Bad request (check model/prompt)", "details": str(e)}
    
    # ✅ NEW: Catch-all for unexpected errors
    except Exception as e:
        return {"success": False, "error": "Unexpected error", "details": str(e)}


# --------------------------------------------------------------
# Test it
# --------------------------------------------------------------
print("🧪 TESTING ERROR HANDLING")
print("="*60)
result = ask_openai_v1("What is 2+2?")
if result["success"]:
    print(f"✅ Success: {result['data']}")
else:
    print(f"❌ Error: {result['error']}")
print("="*60)

### 💡 Key Improvements
- Catches errors instead of crashing
- Returns structured response (dict) with actionable error messages

---

## 🔄 Phase 2: Add Retry Logic

Some errors are temporary. Retry with exponential backoff.

```
Attempt 1: Immediate
Attempt 2: Wait 1 second
Attempt 3: Wait 2 seconds
Attempt 4: Wait 4 seconds
Attempt 5: Wait 8 seconds
```

### Which Errors Should Retry?

✅ **Retry these:** Rate limits, network errors

❌ **Don't retry these:** Authentication errors, invalid requests

In [ ]:
def ask_openai_v2(prompt, max_retries=3):  # ✅ NEW: max_retries parameter
    """
    Version 2: Adds retry logic with exponential backoff.
    
    Returns a dict with:
    - success (bool) 
    - data (str) and attempts (int) when success is True
    - error and details (str) when success is False
    """
    # ✅ NEW: Define which errors are worth retrying
    RETRYABLE_ERRORS = (openai.RateLimitError, openai.APIConnectionError)
    
    # ✅ NEW: Retry loop
    for attempt in range(max_retries):
        try:
            response = client.responses.create(
                model=MODEL,
                input=prompt
            )
            return {
                "success": True, 
                "data": response.output_text.strip(),
                "attempts": attempt + 1  # ✅ NEW: Track attempts
            }
        
        # Permanent error - don't retry
        except openai.AuthenticationError as e:
            return {
                "success": False, 
                "error": "Authentication failed (auth)", 
                "details": str(e)
            }
        
        # Permanent error - don't retry
        except openai.BadRequestError as e:
            return {
                "success": False,
                "error": "Bad request (check model/prompt)",
                "details": str(e)
            }
        
        # ✅ NEW: Transient errors - retry with exponential backoff
        except RETRYABLE_ERRORS as e:
            error_type = "rate_limit" if isinstance(e, openai.RateLimitError) else "network"
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt  # Exponential backoff: 1s, 2s, 4s
                print(f"⏳ Attempt {attempt + 1} failed ({error_type}). Retrying in {wait_time}s...")
                time.sleep(wait_time)
                continue
            else:
                return {
                    "success": False,
                    "error": f"Failed after {max_retries} attempts ({error_type})",
                    "details": str(e)
                }
        
        # Unknown error - don't retry
        except Exception as e:
            return {
                "success": False, 
                "error": "Unexpected error (unknown)", 
                "details": str(e)
            }


# --------------------------------------------------------------
# Test it
# --------------------------------------------------------------
print("🧪 TESTING RETRY LOGIC")
print("="*60)
result = ask_openai_v2("What is the capital of France?", max_retries=3)
if result["success"]:
    print(f"✅ Success after {result['attempts']} attempt(s)")
    print(f"   Answer: {result['data']}")
else:
    print(f"❌ Error: {result['error']}")
print("="*60)

### 💡 Key Improvements

- Only retries transient errors (rate limits, network)
- Uses exponential backoff (1s, 2s, 4s...) and tracks attempts

---

## ✔️ Phase 3: Add Response Validation

A successful API call doesn't guarantee a useful response. Always validate.

### What to Check

- Response isn't empty
- Meets length requirements
- Format is as expected

In [ ]:
def ask_openai_v3(prompt, max_retries=3, min_length=1, max_length=10000):  # ✅ NEW: validation params
    """
    Version 3: Adds response validation.
    
    Returns a dict with:
    - success (bool)
    - data (str), attempts (int), validated (bool) when success is True
    - error and details (str) when success is False
    """
    
    # ✅ NEW: Validation helper
    def _validate_response(text):
        """Validate response quality."""
        if not text or not text.strip():
            return False, "Empty response"
        if len(text) < min_length:
            return False, f"Response too short (min {min_length} chars)"
        if len(text) > max_length:
            return False, f"Response too long (max {max_length} chars)"
        return True, "Valid"
    
    RETRYABLE_ERRORS = (openai.RateLimitError, openai.APIConnectionError)
    
    for attempt in range(max_retries):
        try:
            response = client.responses.create(
                model=MODEL,
                input=prompt
            )
            response_text = response.output_text.strip()
            
            # ✅ NEW: Validate before returning
            is_valid, validation_msg = _validate_response(response_text)
            
            # ✅ NEW: Only return if validation passes
            if is_valid:
                return {
                    "success": True,
                    "data": response_text,
                    "attempts": attempt + 1,
                    "validated": True
                }
            else:
                # ✅ NEW: Invalid response - retry if attempts left
                if attempt < max_retries - 1:
                    wait_time = 2 ** attempt
                    print(f"⚠️ Attempt {attempt + 1}: {validation_msg}. Retrying in {wait_time}s...")
                    time.sleep(wait_time)
                    continue
                else:
                    return {
                        "success": False,
                        "error": f"Validation failed: {validation_msg}",
                        "details": validation_msg,
                        "data": response_text
                    }
        
        # Permanent error - don't retry
        except openai.AuthenticationError as e:
            return {
                "success": False,
                "error": "Authentication failed (auth)",
                "details": str(e)
            }
        
        # Permanent error - don't retry
        except openai.BadRequestError as e:
            return {
                "success": False,
                "error": "Bad request (check model/prompt)",
                "details": str(e)
            }
        
        # Transient error - retry with backoff
        except RETRYABLE_ERRORS as e:
            error_type = "rate_limit" if isinstance(e, openai.RateLimitError) else "network"
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt
                print(f"⏳ Attempt {attempt + 1} failed ({error_type}). Retrying in {wait_time}s...")
                time.sleep(wait_time)
                continue
            else:
                return {
                    "success": False,
                    "error": f"Failed after {max_retries} attempts ({error_type})",
                    "details": str(e)
                }
        
        # Unknown error - don't retry
        except Exception as e:
            return {
                "success": False,
                "error": "Unexpected error (unknown)",
                "details": str(e)
            }


# --------------------------------------------------------------
# Test with validation
# --------------------------------------------------------------
print("🧪 TESTING RESPONSE VALIDATION")
print("="*60)
result = ask_openai_v3("Name 3 colors", min_length=5, max_length=100)
if result["success"]:
    print(f"✅ Success: {result['data']}")
    print(f"   Validated: {result['validated']}")
    print(f"   Attempts: {result['attempts']}")
else:
    print(f"❌ Error: {result['error']}")
print("="*60)

### 💡 Key Improvements

- Validates response length (configurable min/max)
- Retries on validation failure and returns validation status

---

## 📊 Comparing Helper Versions

This table compares the **helper functions** (`ask_openai_v0` → `ask_openai_v3`). Fallbacks are added in your application code, like the email classifier example.

<div style="text-align: left; display: inline-block;">

| Version | Error Handling | Retry Logic | Validation | Production Ready? |
|---------|---------------|-------------|------------|-------------------|
| v0 | ❌ | ❌ | ❌ | ❌ Never |
| v1 | ✅ | ❌ | ❌ | ⚠️ Development only |
| v2 | ✅ | ✅ | ❌ | ⚠️ Internal tools |
| **v3** | **✅** | **✅** | **✅** | **✅ Yes** |

</div>

**Recommendation:** Use `ask_openai_v3` for production. Layer fallback strategies on top in your application code.

---

## 🛟 Phase 4: Fallback Strategies (On Top of v3)

When all retries and validation fail, degrade gracefully instead of crashing.

### Why No `ask_openai_v4`?

Fallbacks are application-specific. An email classifier falls back to `"general"`. A support bot falls back to `"Please try again later"`. A cache-enabled system returns the last known answer. 

There's no universal fallback that works for every use case.

Keep `ask_openai_v3` as your building block. Add fallback logic in your application code.

<div style="text-align: left; display: inline-block;">

| Strategy | Example | When to Use |
|----------|---------|-------------|
| Default Value | Return `"general"` for classification | Non-critical operations |
| Cached Response | Return last known answer | Frequently asked questions |
| Simpler Model | Try `gpt-4o-mini` if `gpt-5-mini` fails | Some answer better than none |
| User Notification | "Please try again later" | Critical operations |

</div>

Next, you'll see this in action with a complete email classifier example, then build your own.

---

## 💪 Practice: Build Your Own Classifier

Complete example — review how v3 + fallback work together, then build your own.

### Reference: Email Classifier

Study this complete working example. Note the pattern:
1. Define `VALID_CATEGORIES` and `FALLBACK`
2. Call `ask_openai_v3()` with retries
3. Validate AI response against allowed categories
4. Return fallback if invalid or failed

In [ ]:
# --------------------------------------------------------------
# COMPLETE EXAMPLE: Email classifier using v3 + Phase 4 fallback
# Study this, then create your own version below
# --------------------------------------------------------------

def robust_email_classifier(email_text):
    """
    Email classifier using v3 error handling.
    Validates AI response against allowed categories.
    """
    VALID_CATEGORIES = ["sales", "technical", "billing", "general"]
    FALLBACK = "general"
    
    prompt = f"""Classify this email into: sales, technical, billing, or general.

Email: {email_text}

Return only the category name."""
    
    result = ask_openai_v3(prompt, max_retries=3)
    
    if result["success"]:
        category = result["data"].lower().strip()
        
        # ✅ Validate against allowed categories
        if category in VALID_CATEGORIES:
            return {"success": True, "category": category, "source": "ai"}
        else:
            # AI returned invalid category - use fallback
            return {"success": True, "category": FALLBACK, "source": "fallback"}
    
    # API failed - use fallback
    return {"success": True, "category": FALLBACK, "source": "fallback"}


# --------------------------------------------------------------
# Test the example
# --------------------------------------------------------------
test_emails = [
    "I'd like to schedule a product demo for our team.",
    "The dashboard is showing error 500.",
    "I was charged twice this month.",
]

print("🧪 COMPLETE EXAMPLE: Email Classifier")
print("="*60)
for email in test_emails:
    result = robust_email_classifier(email)
    print(f"\nEmail: {email[:40]}...")
    print(f"   Category: {result['category']}")
    print(f"   Source: {result['source']}")
print("="*60)

### 💡 What This Example Demonstrates
- Uses v3 for retries and error handling
- Validates AI response against allowed categories
- Falls back to "general" if AI fails or returns invalid category

### 💪 Your Turn: Support Ticket Urgency Classifier

Build your own classifier using the email classifier as a template.

- Categories: `high`, `medium`, `low`
- Fallback: `medium`
- Validation: category in allowed list

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise: Build Your Own Classifier
# --------------------------------------------------------------
# Build a Support Ticket Urgency Classifier using the email classifier as your template.
# 
# Requirements:
# - Valid categories: ["high", "medium", "low"]
# - Fallback: "medium"
# - Validate AI response against valid categories

def robust_support_classifier(ticket_text):
    """Classify support ticket urgency."""
    
    # TODO 1: Define VALID_CATEGORIES and FALLBACK
    
    # TODO 2: Create the prompt (ask AI to classify as high, medium, or low)
    
    # TODO 3: Call ask_openai_v3(prompt, max_retries=3)
    
    # TODO 4: If success, validate category is in VALID_CATEGORIES
    #            Return {"success": True, "category": category, "source": "ai"}
    
    # TODO 5: If invalid or failed, return fallback
    #            Return {"success": True, "category": FALLBACK, "source": "fallback"}
    
    pass  # Remove this when you add your code


# --------------------------------------------------------------
# Test your function
# --------------------------------------------------------------
# test_tickets = [
#     "System is completely down - all users affected!",
#     "Minor typo on the settings page",
#     "Can't login, have a demo in 1 hour",
# ]
# 
# for ticket in test_tickets:
#     result = robust_support_classifier(ticket)
#     print(f"Ticket: {ticket[:40]}...")
#     print(f"   Urgency: {result['category']}")
#     print(f"   Source: {result['source']}\n")

---

## 🎯 Key Takeaways

### What You Learned

**Error Handling:**
- Catch specific exceptions (AuthenticationError, RateLimitError, etc.)
- Return structured error information with actionable messages

**Retry Logic:**
- Use exponential backoff (1s, 2s, 4s...) for transient errors only
- Set reasonable max retries (3-5 attempts)

**Response Validation:**
- Check for empty/malformed responses and length requirements
- Verify expected format and structure

**Fallback Strategies:**
- Provide safe default responses for graceful degradation
- Track source of response (AI vs fallback)

### Quick Reference

**The Pipeline:** Catch errors → Retry transient failures → Validate response → Fallback if needed

---

### 📍 Next Step

**M03C: Prompt Templates & Management** — Build reusable, version-controlled prompts.

---

## 🔧 Troubleshooting

**Retries not working?**
- Check if error is in RETRYABLE list
- Verify exponential backoff calculation (2 ** attempt)
- Add print statements to debug retry flow

**Still getting rate limit errors?**
- Increase wait times between retries
- Reduce number of concurrent requests
- Consider upgrading API tier

**Validation failing unexpectedly?**
- Check validation rules aren't too strict
- Log raw responses to see actual content
- Test with known good responses first

**Fallbacks not activating?**
- Verify fallback value is provided
- Check error is being caught properly
- Test with intentional failures

**High error rates in production?**
- Monitor which errors are most common
- Check OpenAI API status page
- Verify network connectivity is stable

**Unexpected exceptions?**
- Always have a catch-all Exception handler
- Log full exception details
- Return structured error response

---